<a href="https://colab.research.google.com/github/yashsinghal1234/twitter_sentiment_analysis/blob/main/twitter_sentiment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [21]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import re
import nltk
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn.metrics import confusion_matrix, f1_score, classification_report
from sklearn.preprocessing import StandardScaler
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

nltk.download('stopwords')
# Load datasets
train = pd.read_csv('train_tweet.csv')
test = pd.read_csv('test_tweets.csv')

# Check for missing values
print(train.isnull().sum())
print(test.isnull().sum())

id       0
label    0
tweet    0
dtype: int64
id       0
tweet    0
dtype: int64


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [22]:
def clean_tweet(text):
    text = text.lower()  # Lowercase
    text = re.sub(r'[^a-zA-Z\s]', '', text)  # Remove non-alphabetic characters
    text = ' '.join([word for word in text.split() if word not in stopwords.words('english')])  # Remove stopwords
    return text

In [23]:
train['clean_tweet'] = train['tweet'].apply(clean_tweet)
test['clean_tweet'] = test['tweet'].apply(clean_tweet)

In [24]:
# Feature Extraction using TF-IDF
vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))  # Unigrams and bigrams
X = vectorizer.fit_transform(train['clean_tweet']).toarray()
y = train['label']

In [25]:
# Train-test split
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.25, random_state=42)

# Standardization
scaler = StandardScaler(with_mean=False)  # with_mean=False for sparse matrices
X_train = scaler.fit_transform(X_train)
X_valid = scaler.transform(X_valid)

In [26]:
# Model Selection and Hyperparameter Tuning
models = {
    'Random Forest': RandomForestClassifier(),
    'Logistic Regression': LogisticRegression(max_iter=200),
    'Decision Tree': DecisionTreeClassifier(),
    'SVC': SVC(),
    'XGBoost': XGBClassifier(use_label_encoder=False, eval_metric='mlogloss')
}

In [ ]:
for model_name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_valid)
    print(f"{model_name} - Training Accuracy: {model.score(X_train, y_train):.4f}")
    print(f"{model_name} - Validation Accuracy: {model.score(X_valid, y_valid):.4f}")
    print(f"{model_name} - F1 Score: {f1_score(y_valid, y_pred, average='weighted'):.4f}")
    print(f"{model_name} - Confusion Matrix:\n{confusion_matrix(y_valid, y_pred)}\n")

Random Forest - Training Accuracy: 0.9992
Random Forest - Validation Accuracy: 0.9578
Random Forest - F1 Score: 0.9520
Random Forest - Confusion Matrix:
[[7391   41]
 [ 296  263]]

Logistic Regression - Training Accuracy: 0.9974
Logistic Regression - Validation Accuracy: 0.9263
Logistic Regression - F1 Score: 0.9302
Logistic Regression - Confusion Matrix:
[[7065  367]
 [ 222  337]]

Decision Tree - Training Accuracy: 0.9992
Decision Tree - Validation Accuracy: 0.9409
Decision Tree - F1 Score: 0.9393
Decision Tree - Confusion Matrix:
[[7228  204]
 [ 268  291]]



In [ ]:
# Example of Manual Testing
def manual_testing(tweet):
    cleaned_tweet = clean_tweet(tweet)
    tweet_vector = vectorizer.transform([cleaned_tweet])
    tweet_vector = scaler.transform(tweet_vector)
    prediction = model.predict(tweet_vector)
    return prediction

In [28]:
# Test the model with a manual input
test_tweet = input("Enter a tweet for sentiment prediction:\n")
print("Predicted Sentiment:", manual_testing(test_tweet))

Enter a tweet for sentiment prediction:
i like pizza


ValueError: cannot use sparse input in 'SVC' trained on dense data